In [6]:
from langgraph.graph import StateGraph, END

def node_a(state):
    return {"msg": "hello from A"}

def node_b(state):
    return {"msg": state["msg"] + " - B"}

# Graph
graph = StateGraph(dict)

# Add nodes
graph.add_node("A", node_a)
graph.add_node("B", node_b)

# Entry point
graph.set_entry_point("A")

# Edges
graph.add_edge("A", "B")
graph.add_edge("B", END)

# Compile & Run
app = graph.compile()
result = app.invoke({})
print(result)
# Output: {'msg': 'hello from A - B'}

{'msg': 'hello from A - B'}


In [7]:
def router(state):
    if "math" in state["input"]:
        return {"route": "tool"}
    return {"route":"llm"}

def tool_node(state):
    return {"output":"using calculator"}

def llm_node(state):
    return {"output": "using llm"}

graph = StateGraph(dict)

graph.add_node("router",router)
graph.add_node("tool",tool_node)
graph.add_node("llm",llm_node)

graph.set_entry_point("router")

# conditional edge 
graph.add_conditional_edges(
    "router",
    lambda state: state["route"],
    {
        "tool": "tool",
        "llm":"llm"
    }
)

app = graph.compile()
print(app.invoke({"input": "solve math problem"}))

{'output': 'using calculator'}


In [15]:
# from typing import TypedDict

# from langgraph.graph import StateGraph

# class State(TypedDict,tool = False):
#     input: str
#     tool: str
#     log: str
#     result:str

# def router(state):
#     return{}

# def tool_node(state):
#     return {"tool":"Tool executed"}

# def logger_node(state):
#     return {
#         "result": f"{state.get('tool')} | {state.get("log")}"
#     }
# def final_node(state):
#     return{
#         "result":f"{state.get("tool")} | {state.get("log")}"
#     }


# graph = StateGraph(State)

# graph.add_node("router",router)
# graph.add_node("tool",tool_node)
# graph.add_node("final",final_node)

# graph.set_entry_point("router")

# graph.add_conditional_edges(
#     "router",
#     lambda state: ["tool","logger"],
#     {
#         "tool":"tool",
#         "logger":"logger"
#     }
# )

# # merge
# graph.add_node("tool","final")
# graph.add_node("logger","final")

# # compile and run
# app = graph.compile()
# result = app.invoke({"input":"hello"})
# print(result)

SyntaxError: f-string: unmatched '(' (1125641196.py, line 19)

In [16]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

# ── State ─────────────────────────────────────────────────────
# FIX 1: TypedDict mein tool=False nahi hota
class State(TypedDict):
    input : str
    tool  : str
    log   : str
    result: str

# ── Nodes ─────────────────────────────────────────────────────
def router_node(state):
    # FIX 2: router ko decide karna hai — tool ya logger
    if state.get("input") == "hello":
        return {}
    return {}

def tool_node(state):
    return {"tool": "Tool executed"}

def logger_node(state):
    # FIX 3: f-string mein same quote use nahi kar sakte
    return {"log": f"Logger: {state.get('input')}"}

def final_node(state):
    # FIX 4: f-string fix
    return {"result": f"{state.get('tool')} | {state.get('log')}"}

# ── Graph ─────────────────────────────────────────────────────
graph = StateGraph(State)

graph.add_node("router", router_node)
graph.add_node("tool",   tool_node)
graph.add_node("logger", logger_node)
graph.add_node("final",  final_node)

graph.set_entry_point("router")

# FIX 5: conditional edge function ek string return karta hai — list nahi
def route_decision(state):
    return "tool"   # ya "logger" — condition ke hisaab se

graph.add_conditional_edges(
    "router",
    route_decision,
    {
        "tool"  : "tool",
        "logger": "logger"
    }
)

# FIX 6: add_edge use karo — add_node nahi (merge karna hai)
graph.add_edge("tool",   "final")
graph.add_edge("logger", "final")
graph.add_edge("final",  END)

# ── Compile & Run ─────────────────────────────────────────────
app = graph.compile()
result = app.invoke({
    "input" : "hello",
    "tool"  : "",
    "log"   : "",
    "result": ""
})
print(result)
# Output: {'input': 'hello', 'tool': 'Tool executed', 'log': '', 'result': 'Tool executed | None'}

{'input': 'hello', 'tool': 'Tool executed', 'log': '', 'result': 'Tool executed | '}
